In [1]:
import sys
import os
import time
import torch

# 将当前根目录加入寻址路径
sys.path.append(os.getcwd())

from GAME.OthelloGame import OthelloGame
from AGENTS.OthelloPlayers import PureMCTSPlayer
from CORE.SelfPlay import execute_episode
from CNN.NNetTrainer import NNetWrapper

def main():
    print("=" * 60)
    print("Training start...")  # 终端输出：训练开始
    # 终端输出：打印当前 PyTorch 正在使用的设备 (CPU 或 CUDA)
    print(f"PyTorch running device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
    print("=" * 60)

    # 初始化 8x8 黑白棋游戏
    game = OthelloGame(8)

    # ====================================================
    # 阶段 1：数据生成 (Self-Play)
    # ====================================================
    # 终端输出：[阶段 1/3] MCTS 自我对弈生成数据中...
    print("\n[Phase 1/3] Generating data via MCTS self-play...") 
    
    # 思考时间设为 0.5 秒用于快速跑通测试（实际训练时可以加大）
    mcts_generator = PureMCTSPlayer(game, time_limit=0.5) 
    
    # 测试时跑 2 局，确保流程能快速跑通
    num_episodes = 2
    full_dataset = []
    
    for i in range(num_episodes):
        # 终端输出：开始对局...
        print(f"\n>>> Starting episode {i+1}/{num_episodes}...") 
        start_time = time.time()
        
        # 调用 CORE 里的 execute_episode 函数生成单局数据
        episode_data = execute_episode(game, mcts_generator)
        full_dataset.extend(episode_data)
        
        # 终端输出：本局耗时 XXX 秒，产生 XXX 条数据
        print(f"--- Episode elapsed time: {time.time() - start_time:.2f} seconds, generated {len(episode_data)} samples ---")
        
    # 终端输出：数据生成完毕！共收集到 XXX 个训练样本。
    print(f"\nData generation complete! Collected {len(full_dataset)} (s, pi, z) training samples in total.")

    # ====================================================
    # 阶段 2：深度学习训练 (Training)
    # ====================================================
    # 终端输出：[阶段 2/3] 将数据输入 CNN 双头网络进行训练...
    print("\n[Phase 2/3] Feeding data into CNN dual-head network for training...") 
    
    # 调用 CNN 里的 NNetWrapper 初始化网络
    trainer = NNetWrapper(game)
    
    # 开始训练：使用收集到的 full_dataset 跑 10 轮 (Epochs)
    trainer.train(full_dataset, epochs=10, batch_size=32)
    
    # 终端输出：训练阶段完成！Loss 值已经下降！
    print("\nTraining phase complete! Loss value has decreased.") 

    # ====================================================
    # 阶段 3：保存模型 (Saving)
    # ====================================================
    # 终端输出：[阶段 3/3] 保存神经网络模型...
    print("\n[Phase 3/3] Saving neural network model...") 
    
    # 动态获取当前脚本所在目录下的 MODELS 文件夹绝对路径
    models_dir = os.path.join(os.getcwd(), "MODELS")
    
    # 防御性编程：如果 MODELS 文件夹不存在则自动创建，防止保存时崩溃
    if not os.path.exists(models_dir):
        os.makedirs(models_dir)
        # 终端输出：已自动创建模型保存目录
        print(f"Automatically created model saving directory: {models_dir}") 
    
    # 存入指定的模型文件夹
    trainer.save_checkpoint(folder=models_dir, filename="alphazero_v1_test.pth")
    
    print("=" * 60)
    # 终端输出：恭喜！成功跑通了 AlphaZero 的整个生命周期！
    print("Congratulations! You have successfully run the entire AlphaZero lifecycle!") 
    print("=" * 60)

# 标准的 Python 入口保护，解决之前代码由于缺少函数体导致的缩进报错问题
if __name__ == "__main__":
    main()

Training start...
PyTorch running device: cuda

[Phase 1/3] Generating data via MCTS self-play...

>>> Starting episode 1/2...
thinking ...
Episode finished. Total steps: 61
--- Episode elapsed time: 30.86 seconds, generated 61 samples ---

>>> Starting episode 2/2...
thinking ...
Episode finished. Total steps: 60
--- Episode elapsed time: 30.35 seconds, generated 60 samples ---

Data generation complete! Collected 121 (s, pi, z) training samples in total.

[Phase 2/3] Feeding data into CNN dual-head network for training...
Starting training on device: cuda
Epoch 01/10 | Policy Loss: 4.1228 | Value Loss: 0.4453
Epoch 02/10 | Policy Loss: 3.2083 | Value Loss: 0.0393
Epoch 03/10 | Policy Loss: 2.7713 | Value Loss: 0.0327
Epoch 04/10 | Policy Loss: 2.4941 | Value Loss: 0.0161
Epoch 05/10 | Policy Loss: 2.2891 | Value Loss: 0.0175
Epoch 06/10 | Policy Loss: 2.1143 | Value Loss: 0.0054
Epoch 07/10 | Policy Loss: 1.9930 | Value Loss: 0.0266
Epoch 08/10 | Policy Loss: 1.8978 | Value Loss: 0.0

In [1]:
import sys
import os
import torch
import numpy as np

# 确保寻址正确
sys.path.append(os.getcwd())

from GAME.OthelloGame import OthelloGame
from AGENTS.OthelloPlayers import PureMCTSPlayer, AlphaZeroMCTS
from CNN.NNetTrainer import NNetWrapper
from CORE.Arena import Arena

if __name__ == "__main__":
    print("="*60)
    print("AlphaZero model test starting...")
    print("="*60)

    # 1. 初始化游戏
    game = OthelloGame(8)

    # 2. 召唤旧时代的残党：纯 MCTS (每步思考 0.5 秒)
    print("[1/4] Loading legacy PureMCTS...")
    old_mcts = PureMCTSPlayer(game, time_limit=0.5)

    # 3. 唤醒新时代的恶魔：加载你刚刚训练的神经网络
    print("[2/4] Loading trained Neural Network from MODELS folder...")
    nnet = NNetWrapper(game)
    # 【注意】根据你的环境，这里可能需要改成 os.getcwd()
    models_dir = os.path.join(os.getcwd(), "MODELS")
    nnet.load_checkpoint(folder=models_dir, filename="alphazero_v1_test.pth")

    # 4. 把神经网络装进新的躯体 (每次思考只推演 50 步，速度极快)
    print("[3/4] Implanting Neural Network into AlphaZeroMCTS...")
    new_mcts = AlphaZeroMCTS(game, nnet, num_sims=50, c_puct=1.0)

    # 5. 建立角斗场
    print("[4/4] Setting up the Arena...")
    arena = Arena(player1=new_mcts, player2=old_mcts, game=game)

    # 6. 开战！(交替先手打 2 局测试流程)
    # 比赛中开启 verbose=True 可以看到每步下在哪
    print("\n⚔️ BATTLE START: AlphaZero (P1) VS PureMCTS (P2) ⚔️")
    p1_wins, p2_wins, draws = arena.play_games(num_games=2, verbose=False)

    print("\n" + "="*50)
    print(f" BATTLE RESULTS:")
    print(f"  AlphaZero Wins : {p1_wins}")
    print(f"  PureMCTS Wins  : {p2_wins}")
    print(f"  Draws          : {draws}")
    print("="*50)

AlphaZero model test starting...
[1/4] Loading legacy PureMCTS...
[2/4] Loading trained Neural Network from MODELS folder...
Model loaded: /home3/tszj13/final work/MODELS/alphazero_v1_test.pth
[3/4] Implanting Neural Network into AlphaZeroMCTS...
[4/4] Setting up the Arena...

⚔️ BATTLE START: AlphaZero (P1) VS PureMCTS (P2) ⚔️
========== Evaluation Start: Player 1 vs Player 2 (2 Games) ==========

[First Half] Player 1 (Black/First) vs Player 2 (White)
Game 1/1 finished -> Winner: Player 1

[Second Half] Player 2 (Black/First) vs Player 1 (White)
Game 2/2 finished -> Winner: Player 1

================== Final Results ==================
Player 1 Wins: 2
Player 2 Wins: 0
Draws: 0


 BATTLE RESULTS:
  AlphaZero Wins : 2
  PureMCTS Wins  : 0
  Draws          : 0
